# Policy-Aware RAG Smoke Tests

Use this notebook to run a small set of positive and negative policy cases against the deployed Function App. Each case sends one request with an ODRL 2.2 JSON-LD policy and prints the final outcome so you can compare allow and deny behavior quickly.

The gateway now derives Cosmos DB security filters from the policy on the server side, so this notebook does not send manual retrieval filters. The response payload also includes a `decisionGraph` when the orchestration reaches the multi-agent decision stage.


In [3]:
import json
import os
import time
import requests

# For local testing
FUNCTION_APP_URL = "http://localhost:7071/api/orchestrators/start"
FUNCTION_APP_KEY = ""

# For function app testing
# FUNCTION_APP_URL = os.environ.get("FUNCTION_APP_URL")
# FUNCTION_APP_KEY = os.environ.get("FUNCTION_APP_KEY")

POLICY_DIR = os.environ.get("ODRL_POLICY_DIR")
COSMOS_ENRON_COLLECTION = os.environ.get("COSMOSDB_ENRON_COLLECTION")
ORCHESTRATION_TIMEOUT_SECONDS = int(os.environ.get("ORCHESTRATION_TIMEOUT_SECONDS"))


def load_policy(policy_filename: str) -> dict:
    """Load an ODRL policy document from the configured policy directory.

    Args:
        policy_filename: Relative or absolute path to the policy file.

    Returns:
        The parsed ODRL policy document.
    """
    policy_path = policy_filename
    if not os.path.isabs(policy_path):
        policy_path = os.path.join(POLICY_DIR, policy_filename)
    with open(policy_path, "r", encoding="utf-8") as policy_file:
        return json.load(policy_file)


def build_payload(case: dict) -> dict:
    """Build a function app request payload for a notebook case.

    Args:
        case: Notebook case definition.

    Returns:
        A JSON-serializable payload for the orchestration endpoint.
    """
    return {
        "principal": case["principal"],
        "odrl_policy": load_policy(case["policy_filename"]),
        "query_text": case["query_text"],
        "query_embedding": case.get("query_embedding", [0.1, 0.2, 0.3]),
        "action": case.get("action", "summarise"),
        "cosmos_collection": case.get("cosmos_collection", COSMOS_ENRON_COLLECTION),
    }


def start_and_wait(payload: dict) -> dict:
    """Start the orchestration and poll until it reaches a terminal state.

    Args:
        payload: Orchestration request payload.

    Returns:
        A dictionary with the start response, polling URL, and terminal status.
    """
    headers = {
                "x-functions-key": FUNCTION_APP_KEY,
                "Content-Type": "application/json"
               }
    response = requests.post(FUNCTION_APP_URL, json=payload, headers=headers, timeout=ORCHESTRATION_TIMEOUT_SECONDS)
    result = {
        "start_status": response.status_code,
        "status_url": None,
        "start_body": None,
        "final": None,
        "raw_start": response.text,
    }

    try:
        result["start_body"] = response.json()
    except Exception:
        result["start_body"] = None

    status_url = None
    if isinstance(result["start_body"], dict):
        status_url = result["start_body"].get("statusQueryGetUri")
    if not status_url:
        status_url = response.headers.get("Location")
    result["status_url"] = status_url

    if not status_url:
        return result

    deadline = time.time() + ORCHESTRATION_TIMEOUT_SECONDS
    terminal_statuses = {"Completed", "Failed", "Terminated", "Canceled"}
    while time.time() < deadline:
        status_resp = requests.get(status_url, timeout=ORCHESTRATION_TIMEOUT_SECONDS)
        try:
            status_json = status_resp.json()
        except Exception:
            result["final"] = {"runtimeStatus": "NonJSON", "raw": status_resp.text}
            return result

        runtime_status = status_json.get("runtimeStatus")
        if runtime_status in terminal_statuses:
            result["final"] = status_json
            return result

        time.sleep(1)

    raise TimeoutError(f"Timed out waiting for orchestration after {ORCHESTRATION_TIMEOUT_SECONDS} seconds")


CASES = [
    {
        "name": "Positive: governance admin counts emails from Fran Fagan",
        "policy_filename": "30-full-access.json",
        "principal": {
            "userId": "notebook-user",
            "role": "pii-data-governance-admin",
            "declaredIntent": "business_review",
        },
        "query_text": "How many emails did Fran Fagan send?",
        "action": "summarise",
        "expected_statuses": ["count_result"],
    },
    {
        "name": "Positive: privacy compliance analyst allows compliance review",
        "policy_filename": "20-privacy-analyst.json",
        "principal": {
            "userId": "notebook-user",
            "role": "privacy-compliance-analyst",
            "declaredIntent": "compliance_review",
        },
        "query_text": "Summarise the relevant privacy review emails",
        "action": "summarise",
        "expected_statuses": ["ok", "no_results"],
    },
    {
        "name": "Positive: business observer handles routing and triage",
        "policy_filename": "00-no-pii-observer.json",
        "principal": {
            "userId": "notebook-user",
            "role": "business-observer",
            "declaredIntent": "routing",
        },
        "query_text": "Summarise the emails for routing and triage",
        "action": "summarise",
        "expected_statuses": ["ok", "no_results"],
    },
    {
        "name": "Positive: metadata-role match returns eligible records",
        "policy_filename": "20-privacy-analyst.json",
        "principal": {
            "userId": "notebook-user",
            "role": "privacy-compliance-analyst",
            "declaredIntent": "compliance_review",
        },
        "query_text": "Summarise the relevant privacy review emails",
        "action": "summarise",
        "expected_statuses": ["ok", "no_results"],
    },
    {
        "name": "Negative: metadata-role mismatch filters out records for the same query",
        "policy_filename": "20-privacy-analyst.json",
        "principal": {
            "userId": "notebook-user",
            "role": "customer-support-specialist",
            "declaredIntent": "compliance_review",
        },
        "query_text": "Summarise the relevant privacy review emails",
        "action": "summarise",
        "expected_statuses": ["ok", "no_results"],
    },
    {
        "name": "Negative: customer support specialist gets denied for export",
        "policy_filename": "10-support-limited.json",
        "principal": {
            "userId": "notebook-user",
            "role": "customer-support-specialist",
            "declaredIntent": "sales_followup",
        },
        "query_text": "Export the email archive",
        "action": "export",
        "expected_statuses": ["denied"],
    },
]

print("FUNCTION_APP_URL:", FUNCTION_APP_URL)
print("FUNCTION_APP_KEY:", "<hidden>" if FUNCTION_APP_KEY else "<not set>")
print("POLICY_DIR:", POLICY_DIR)
print("COSMOS_ENRON_COLLECTION:", COSMOS_ENRON_COLLECTION)
print("Cases configured:", len(CASES))
for index, case in enumerate(CASES, start=1):
    print(f"  {index}. {case['name']} -> expected {', '.join(case['expected_statuses'])}")


FUNCTION_APP_URL: http://localhost:7071/api/orchestrators/start
FUNCTION_APP_KEY: <not set>
POLICY_DIR: ../odrl_policies
COSMOS_ENRON_COLLECTION: EnronEmailVectorStore
Cases configured: 6
  1. Positive: governance admin counts emails from Fran Fagan -> expected count_result
  2. Positive: privacy compliance analyst allows compliance review -> expected ok, no_results
  3. Positive: business observer handles routing and triage -> expected ok, no_results
  4. Positive: metadata-role match returns eligible records -> expected ok, no_results
  5. Negative: metadata-role mismatch filters out records for the same query -> expected ok, no_results
  6. Negative: customer support specialist gets denied for export -> expected denied


In [4]:
results = []

for index, case in enumerate(CASES, start=1):
    print(f"\n=== Case {index}: {case['name']} ===")
    payload = build_payload(case)
    print("Request payload:")
    print(json.dumps(
        {
            "principal": payload["principal"],
            "query_text": payload["query_text"],
            "action": payload["action"],
            "cosmos_collection": payload["cosmos_collection"],
            "policy_uid": payload["odrl_policy"].get("uid"),
        },
        indent=2,
    ))

    result = start_and_wait(payload)
    final = result.get("final") or {}
    output = final.get("output") or {}

    print("Start status:", result.get("start_status"))
    if result.get("status_url"):
        print("Status URL:", result.get("status_url"))
    print("runtimeStatus:", final.get("runtimeStatus"))
    print("Outcome status:", output.get("status"))
    if output.get("outcomeType") is not None:
        print("Outcome type:", output.get("outcomeType"))
    if output.get("result") is not None:
        print("Result:", output.get("result"))
    if output.get("reason") is not None:
        print("Reason:", output.get("reason"))
    if output.get("decisionGraph") is not None:
        print("Decision graph:")
        print(json.dumps(output.get("decisionGraph"), indent=2))

    observed_status = output.get("status")
    expected_statuses = case["expected_statuses"]
    passed = observed_status in expected_statuses
    results.append({"name": case["name"], "passed": passed, "observed": observed_status})
    print("Test result:", "PASS" if passed else f"FAIL (expected one of {expected_statuses})")

print("\n=== Summary ===")
for item in results:
    print(f"{item['name']}: {'PASS' if item['passed'] else 'FAIL'} ({item['observed']})")



=== Case 1: Positive: governance admin counts emails from Fran Fagan ===
Request payload:
{
  "principal": {
    "userId": "notebook-user",
    "role": "pii-data-governance-admin",
    "declaredIntent": "business_review"
  },
  "query_text": "How many emails did Fran Fagan send?",
  "action": "summarise",
  "cosmos_collection": "EnronEmailVectorStore",
  "policy_uid": "urn:policyaware:policy:pii-data-governance-admin"
}
Start status: 202
Status URL: http://localhost:7071/runtime/webhooks/durabletask/instances/e8864b5ccd7744c79a005d4445fdab4b?taskHub=TestHubName&connection=Storage&code=hX21aSzXgEUONk4v_iJS0tQnxA4Fid4ygnOEOHUjuaIxAzFuzK92qw==
runtimeStatus: Completed
Outcome status: ok
Outcome type: count_result
Result: Fran Fagan sent 14 emails.
Test result: FAIL (expected one of ['count_result'])

=== Case 2: Positive: privacy compliance analyst allows compliance review ===
Request payload:
{
  "principal": {
    "userId": "notebook-user",
    "role": "privacy-compliance-analyst",
   